In [0]:
from pyspark.sql.functions import *
from pyspark.sql import *
from pyspark import *

In [0]:
# Parquet is the default file format within Apache Spark.

df_nlp = spark.read.text('/Volumes/prodsqldwh/adls/nlp_files/*.txt')


In [0]:
display(df_nlp.first())

In [0]:
display(df_nlp.count())

In [0]:
df_nlp.show(5, truncate=False)
display(df_nlp.limit(5))
df_nlp.limit(5).show(truncate=False)

In [0]:
df_nlp.printSchema()

In [0]:
df_nlp_lower = df_nlp.select(lower(df_nlp.value).alias('LowerValue'))
df_nlp_upper = df_nlp.select(upper(df_nlp.value).alias('UpperValue'))

df_nlp_lower.show(5, truncate= False)
df_nlp_upper.show(5, truncate= False)

In [0]:
#RegExp:
df_nlp_regex = df_nlp.select(regexp_replace(df_nlp.value, ' ', '_').alias('BlankToUnderscore'))
df_nlp_regex2 = df_nlp.select(regexp_replace(df_nlp.value, ' ', '').alias('BlankTonone'))

df_nlp_regex.show(5, truncate=False)
df_nlp_regex2.show(5, truncate=False)

In [0]:
df_nlp_split = df_nlp.select(split(df_nlp.value, ' ').alias('Words'))
#Remove punctuations or special chars:
df_nlp_split2 = df_nlp.select(split(df_nlp.value, '[ .,:;—?-]').alias('Words'))

df_nlp_split.show(10, truncate=False)
df_nlp_split2.show(10, truncate=False)

In [0]:
df_nlp_wordbyrow = df_nlp_split2.select(explode(df_nlp_split2.Words).alias('word'))
df_nlp_wordbyrow = df_nlp_wordbyrow.where(df_nlp_wordbyrow.word != '')
#df_nlp_wordbyrow = df_nlp_wordbyrow.where(length(df_nlp_wordbyrow.word) > 0)

df_nlp_wordbyrow.show(100, truncate=False)

In [0]:
display(df_nlp.count())

In [0]:
display(df_nlp_wordbyrow.count())

In [0]:
df_nlp_wordbyrow2 = (
df_nlp_wordbyrow.select(monotonically_increasing_id().alias('WordNo'),
                        col('word'), 
                        length(col('word')).alias('lettersNo')
                        )
                        .where(col('lettersNo') > 1)
)
df_nlp_wordbyrow2.show(truncate=False)        


In [0]:
df_nlp_partition = (
    df_nlp_wordbyrow2.withColumn('TextPart',
                                 when(df_nlp_wordbyrow2.WordNo < 60000, 0) 
                                 .when(df_nlp_wordbyrow2.WordNo < 120000, 1) 
                                 .when(df_nlp_wordbyrow2.WordNo < 180000, 2)
                                 .otherwise(3)              
                                )
)
df_nlp_partition.show()

In [0]:
df_nlp_partition = df_nlp_partition.repartition(4, 'TextPart')
#df_nlp_partition.rdd.getNumPartitions()

In [0]:
#Moving Window Query Analysis: 

(
df_nlp_partition.select(col('WordNo'),
                        df_nlp_partition.word.alias('w1'),
                        lead(df_nlp_partition.word, 1).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w2'),
                        lead(df_nlp_partition.word, 2).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w3'),
                        col('TextPart')
                        )
                .orderBy(col('WordNo').asc())
).show()


In [0]:
(
df_nlp_partition.select(col('WordNo'),
                        df_nlp_partition.word.alias('w1'),
                        lag(df_nlp_partition.word, 1).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w2'),
                        lag(df_nlp_partition.word, 2).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w3'),
                        col('TextPart')
                        )
                .where(df_nlp_partition.TextPart == 2)
                .orderBy(col('WordNo').asc())
).show()


In [0]:
df_nlp_partition_Lag_lead = (
df_nlp_partition.select(col('WordNo'),
                        lag(df_nlp_partition.word, 2).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w1'),
                        lag(df_nlp_partition.word, 1).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w2'),
                        df_nlp_partition.word.alias('w3'),
                        lead(df_nlp_partition.word, 1).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w4'),
                        lead(df_nlp_partition.word, 2).over(Window.partitionBy(df_nlp_partition.TextPart).orderBy(df_nlp_partition.WordNo.asc())).alias('w5'),
                        col('TextPart')
                        )
                .groupBy(col('w1'), col('w2'), col('w3'), col('w4'), col('w5'))
                .agg(count('*').alias('SequenceWordsCount'))
                .orderBy(col('SequenceWordsCount').desc())
)

df_nlp_partition_Lag_lead.show()

In [0]:
df_nlp_partition_Lag_lead.createOrReplaceTempView('word_sequences')

In [0]:
#Common word Sequences: 

query = """
SELECT w1, 
       w2, 
       w3, 
       w4, 
       w5, 
       COUNT(*) AS SequenceWordsCount,
       FIRST(Textpart) AS FirstTextpart,
       LAST(Textpart) AS LastTextpart
FROM word_sequences
GROUP BY w1, w2, w3, w4, w5
ORDER BY SequenceWordsCount DESC
"""

spark.sql(query).show(truncate=False)
